# Phát hiện Gian lận Thẻ tín dụng bằng Cây quyết định (Decision Tree)

Notebook này thực hiện trọn vẹn quy trình xây dựng mô hình **Cây quyết định (`DecisionTreeClassifier`)** trên bộ dữ liệu giao dịch `train.csv` và xuất tệp dự đoán `submission.csv` đúng theo định dạng của `sample_submission.csv` (sử dụng `id` từ `test.csv`).

### Quy trình thực hiện
1. **Đọc và kiểm tra cấu trúc dữ liệu** (`train.csv`, `test.csv`, `sample_submission.csv`).
2. **Khám phá dữ liệu (EDA):** Phân tích mức độ mất cân bằng lớp và các tín hiệu rủi ro chính.
3. **Kỹ thuật đặc trưng & Tiền xử lý:** Tạo thêm đặc trưng tổng hợp phù hợp với cơ chế chia nhánh trực giao của Cây quyết định và mã hóa One-Hot các biến phân loại.
4. **Kiểm chứng chéo (Stratified K-Fold CV):** So sánh ảnh hưởng của đặc trưng mới và tham số chống quá khớp (`max_depth`, `min_samples_leaf`).
5. **Huấn luyện mô hình cuối cùng & Diễn giải:** Phân tích độ quan trọng đặc trưng và các luật chia nhánh hàng đầu của cây.
6. **Xuất và kiểm định `submission.csv`:** Kiểm tra đầy đủ số dòng, cột, `id` và miền giá trị xác suất.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore")

# Tự động xác định thư mục chứa dữ liệu (chạy được cả ở PCCF/ hoặc thư mục gốc)
DATA_DIR = "data" if os.path.exists("data/train.csv") else os.path.join("PCCF", "data")
OUTPUT_PATH = "submission.csv" if os.path.exists("data/train.csv") else os.path.join("PCCF", "submission.csv")

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print(f"Kích thước train_df : {train_df.shape}")
print(f"Kích thước test_df  : {test_df.shape}")
print(f"Kích thước sample_sub: {sample_sub.shape}")
print(f"Tổng số ô bị thiếu (NaN) trong train: {train_df.isna().sum().sum()}")
print(f"Tổng số ô bị thiếu (NaN) trong test : {test_df.isna().sum().sum()}")
print("\n5 dòng đầu của sample_submission.csv:")
print(sample_sub.head())

Kích thước train_df : (699960, 26)
Kích thước test_df  : (299984, 25)
Kích thước sample_sub: (299984, 2)
Tổng số ô bị thiếu (NaN) trong train: 0
Tổng số ô bị thiếu (NaN) trong test : 0

5 dòng đầu của sample_submission.csv:
       id  is_fraud
0  699960    0.0167
1  699961    0.0167
2  699962    0.0167
3  699963    0.0167
4  699964    0.0167


## 1. Khám phá Dữ liệu (EDA) & Hiểu đúng định dạng đầu ra

Quan sát `sample_submission.csv`, ta thấy cột `is_fraud` chứa giá trị thực `0.0167` (tương ứng đúng tỷ lệ gian lận trung bình của tập huấn luyện). Điều này khẳng định bài toán yêu cầu **dự đoán xác suất giao dịch là gian lận** `P(is_fraud = 1 | x)` nằm trong đoạn `[0, 1]` bằng `predict_proba(...)[:, 1]`, chứ không phải nhãn cứng `0/1`.

Ngoài ra, vì dữ liệu bị **mất cân bằng lớp rất mạnh** (~1.67% mẫu dương), ta đánh giá mô hình bằng 3 chỉ số chuẩn cho xác suất trên dữ liệu mất cân bằng:
- **ROC-AUC:** Đo khả năng xếp hạng giao dịch gian lận cao hơn giao dịch bình thường.
- **PR-AUC (Average Precision):** Nhạy hơn với lớp thiểu số (gian lận).
- **LogLoss:** Đo độ chính xác/tin cậy của xác suất dự đoán.

In [2]:
target_counts = train_df["is_fraud"].value_counts()
fraud_rate = train_df["is_fraud"].mean()

print("Phân phối nhãn is_fraud trong tập train:")
print(target_counts)
print(f"Tỷ lệ gian lận trung bình: {fraud_rate:.6f} ({fraud_rate * 100:.2f}%)")
print(f"id của test_df khớp 100% với sample_submission: {(test_df['id'] == sample_sub['id']).all()}")

# Thống kê tỷ lệ gian lận theo các cờ nhị phân (boolean risk flags)
bool_cols = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]

print("\nTỷ lệ gian lận theo từng cờ rủi ro:")
for col in bool_cols:
    stats = train_df.groupby(col)["is_fraud"].agg(["count", "mean"])
    print(f"- {col:30s}: False -> {stats.loc[False, 'mean']*100:5.2f}% | True -> {stats.loc[True, 'mean']*100:5.2f}%")

Phân phối nhãn is_fraud trong tập train:
is_fraud
0.0    688271
1.0     11689
Name: count, dtype: int64
Tỷ lệ gian lận trung bình: 0.016700 (1.67%)
id của test_df khớp 100% với sample_submission: True

Tỷ lệ gian lận theo từng cờ rủi ro:
- is_foreign_transaction        : False ->  1.48% | True -> 12.57%
- is_new_merchant               : False ->  1.12% | True ->  4.73%
- used_vpn                      : False ->  1.50% | True ->  6.55%
- ip_country_mismatch           : False ->  1.43% | True -> 16.50%
- billing_shipping_mismatch     : False ->  1.52% | True -> 12.99%
- is_ai_generated_scam_attempt  : False ->  1.63% | True -> 15.22%


## 2. Kỹ thuật Đặc trưng (Feature Engineering) & Tiền xử lý

### Tại sao cần tạo thêm đặc trưng cho Cây quyết định?
Một **Cây quyết định (`DecisionTreeClassifier`)** chia không gian dữ liệu bằng các lát cắt **vuông góc từng trục** (dạng `x_j <= ngưỡng`). Vì vậy, cây rất khó nắm bắt trực tiếp các mối quan hệ dạng **tổng số cờ cảnh báo** hoặc **tỷ lệ giữa hai cột**.

Ta bổ sung 3 đặc trưng giàu ý nghĩa nghiệp vụ:
1. **`risk_flag_count`**: Tổng số cờ rủi ro cùng lúc bật `True` trên một giao dịch (từ 0 đến 6). Một giao dịch vừa sai lệch quốc gia IP, vừa dùng VPN, vừa sai địa chỉ thanh toán/giao hàng sẽ có mức rủi ro cộng hưởng cao hơn hẳn.
2. **`amount_to_balance_ratio`**: Tỷ lệ giữa số tiền giao dịch (`amount_usd`) và số dư tài khoản (`account_balance_usd`). Giao dịch rút gần cạn hoặc vượt quá mức bình thường của tài khoản là dấu hiệu đáng ngờ điển hình.
3. **`is_night_txn`**: Đánh dấu giao dịch diễn ra vào khung giờ khuya từ `0h` đến `5h` sáng (`time_of_day_hour`).

### Mã hóa biến phân loại
5 cột dạng chuỗi (`merchant_category`, `card_type`, `auth_method`, `channel`, `device_type`) được biến đổi qua `OneHotEncoder(handle_unknown="ignore")`, còn các cột số giữ nguyên (`passthrough`) vì Cây quyết định bất biến với phép co giãn đơn điệu (không cần chuẩn hóa Z-score).

In [3]:
BOOL_COLS = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]
CAT_COLS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]


def build_features(df: pd.DataFrame, use_engineered: bool = True) -> pd.DataFrame:
    """Tạo tập đặc trưng sạch từ dữ liệu thô, loại bỏ id và biến mục tiêu."""
    drop_cols = [c for c in ["id", "is_fraud"] if c in df.columns]
    out = df.drop(columns=drop_cols).copy()

    for col in BOOL_COLS:
        out[col] = out[col].astype(int)

    if use_engineered:
        out["risk_flag_count"] = out[BOOL_COLS].sum(axis=1)
        out["amount_to_balance_ratio"] = out["amount_usd"] / (out["account_balance_usd"] + 1e-6)
        out["is_night_txn"] = ((out["time_of_day_hour"] >= 0) & (out["time_of_day_hour"] <= 5)).astype(int)

    return out


def make_preprocessor(feature_df: pd.DataFrame) -> ColumnTransformer:
    """Tạo bộ tiền xử lý One-Hot cho cột phân loại và giữ nguyên cột số."""
    num_cols = [c for c in feature_df.columns if c not in CAT_COLS]
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
            ("num", "passthrough", num_cols),
        ]
    )


y = train_df["is_fraud"].astype(int)
X_raw = build_features(train_df, use_engineered=False)
X_fe = build_features(train_df, use_engineered=True)
X_test_fe = build_features(test_df, use_engineered=True)

print(f"Số đặc trưng trước Feature Engineering: {X_raw.shape[1]}")
print(f"Số đặc trưng sau Feature Engineering  : {X_fe.shape[1]}")

Số đặc trưng trước Feature Engineering: 24
Số đặc trưng sau Feature Engineering  : 27


## 3. Kiểm chứng chéo (Stratified 3-Fold CV) & Chống quá khớp

Để chọn cấu hình tối ưu cho `DecisionTreeClassifier` mà không bị học vẹt (overfitting) trên gần 700.000 dòng, ta dùng **Stratified 3-Fold Cross-Validation** (giữ nguyên tỷ lệ gian lận ~1.67% ở mỗi fold) để so sánh 3 kịch bản:
1. **Cây cơ sở (không có đặc trưng mới):** `max_depth=9, min_samples_leaf=200`.
2. **Cây đề xuất (có đặc trưng mới + cắt tỉa hợp lý):** `max_depth=9, min_samples_leaf=200, criterion='gini'`.
3. **Cây quá sâu (dễ quá khớp):** `max_depth=12, min_samples_leaf=50` — cho phép nút lá chỉ còn tối thiểu 50 mẫu, khiến xác suất dự đoán ở lá bị nhiễu mạnh và làm tăng `LogLoss`.

In [4]:
def evaluate_decision_tree_cv(X_data: pd.DataFrame, y_data: pd.Series, **tree_params) -> dict:
    """Đánh giá DecisionTreeClassifier bằng Stratified 3-Fold CV."""
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    preprocessor = make_preprocessor(X_data)

    aucs, pr_aucs, losses = [], [], []
    for train_idx, val_idx in skf.split(X_data, y_data):
        pipe = Pipeline([
            ("preprocess", preprocessor),
            ("model", DecisionTreeClassifier(random_state=42, **tree_params)),
        ])
        pipe.fit(X_data.iloc[train_idx], y_data.iloc[train_idx])
        probs = pipe.predict_proba(X_data.iloc[val_idx])[:, 1]

        aucs.append(roc_auc_score(y_data.iloc[val_idx], probs))
        pr_aucs.append(average_precision_score(y_data.iloc[val_idx], probs))
        losses.append(log_loss(y_data.iloc[val_idx], probs))

    return {
        "ROC-AUC": float(np.mean(aucs)),
        "PR-AUC": float(np.mean(pr_aucs)),
        "LogLoss": float(np.mean(losses)),
    }


experiments = [
    ("1. Cơ sở (chưa có FE, d=9, msl=200)", X_raw, {"criterion": "gini", "max_depth": 9, "min_samples_leaf": 200}),
    ("2. Đề xuất (có FE, d=9, msl=200)", X_fe, {"criterion": "gini", "max_depth": 9, "min_samples_leaf": 200}),
    ("3. Cây sâu quá khớp (có FE, d=12, msl=50)", X_fe, {"criterion": "gini", "max_depth": 12, "min_samples_leaf": 50}),
]

results = []
for name, X_cur, params in experiments:
    metrics = evaluate_decision_tree_cv(X_cur, y, **params)
    results.append({"Cấu hình": name, **metrics})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False, float_format=lambda v: f"{v:.5f}"))

                                 Cấu hình  ROC-AUC  PR-AUC  LogLoss
      1. Cơ sở (chưa có FE, d=9, msl=200)  0.90737 0.34159  0.05609
         2. Đề xuất (có FE, d=9, msl=200)  0.92502 0.39944  0.05252
3. Cây sâu quá khớp (có FE, d=12, msl=50)  0.89070 0.40977  0.07904


## 4. Huấn luyện Mô hình Chính thức & Diễn giải Luật quyết định

Kết quả kiểm chứng chéo cho thấy:
- Việc bổ sung 3 đặc trưng mới giúp **ROC-AUC tăng rõ rệt (~0.907 lên ~0.925)** và **PR-AUC tăng mạnh (~0.342 lên ~0.399)**, đồng thời giảm `LogLoss`.
- Khi tăng độ sâu lên `max_depth=12` và hạ `min_samples_leaf=50`, mô hình bắt đầu **quá khớp**: ROC-AUC sụt giảm và LogLoss tăng vọt.

Bây giờ ta huấn luyện **Cây quyết định đề xuất (`criterion='gini', max_depth=9, min_samples_leaf=200`)** trên toàn bộ 699.960 mẫu huấn luyện, đồng thời kiểm tra **Top 10 đặc trưng quan trọng nhất** và in ra 3 tầng chia nhánh đầu tiên của cây để kiểm chứng tính hợp lý.

In [5]:
final_model = Pipeline([
    ("preprocess", make_preprocessor(X_fe)),
    ("model", DecisionTreeClassifier(
        criterion="gini",
        max_depth=9,
        min_samples_leaf=200,
        random_state=42,
    )),
])

final_model.fit(X_fe, y)

# Trích xuất độ quan trọng đặc trưng (Feature Importance theo mức giảm Gini)
feature_names = final_model.named_steps["preprocess"].get_feature_names_out()
importances = final_model.named_steps["model"].feature_importances_
imp_df = (
    pd.DataFrame({"Đặc trưng": feature_names, "Độ quan trọng": importances})
    .sort_values("Độ quan trọng", ascending=False)
    .reset_index(drop=True)
)

print("Top 10 đặc trưng quan trọng nhất:")
print(imp_df.head(10).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\nCấu trúc 3 tầng đầu tiên của Cây quyết định:")
tree_rules = export_text(
    final_model.named_steps["model"],
    feature_names=list(feature_names),
    max_depth=3,
)
print(tree_rules)

Top 10 đặc trưng quan trọng nhất:
                         Đặc trưng  Độ quan trọng
              num__risk_flag_count         0.2907
              num__cvv_retry_count         0.2365
          num__merchant_risk_score         0.1906
               num__velocity_score         0.1310
cat__auth_method_No Authentication         0.1272
                 num__customer_age         0.0200
                   num__amount_usd         0.0010
        num__distance_from_home_km         0.0009
  cat__merchant_category_Streaming         0.0007
         num__hours_since_last_txn         0.0007

Cấu trúc 3 tầng đầu tiên của Cây quyết định:
|--- num__risk_flag_count <= 1.50
|   |--- num__cvv_retry_count <= 0.50
|   |   |--- num__merchant_risk_score <= 61.95
|   |   |   |--- cat__auth_method_No Authentication <= 0.50
|   |   |   |   |--- truncated branch of depth 6
|   |   |   |--- cat__auth_method_No Authentication >  0.50
|   |   |   |   |--- truncated branch of depth 6
|   |   |--- num__merchant_risk_s

## 5. Dự đoán trên tập Kiểm thử và Xuất tệp `submission.csv`

Cuối cùng, ta sử dụng mô hình đã huấn luyện để dự đoán xác suất gian lận `predict_proba(X_test_fe)[:, 1]`, gắn đúng cột `id` từ `test.csv` và lưu ra tệp `submission.csv` theo đúng chuẩn `sample_submission.csv`, sau đó chạy các bước kiểm tra ràng buộc tự động.

In [6]:
test_probs = final_model.predict_proba(X_test_fe)[:, 1]

submission = pd.DataFrame({
    "id": test_df["id"],
    "is_fraud": test_probs,
})

submission.to_csv(OUTPUT_PATH, index=False)

# Kiểm tra hợp lệ tệp vừa xuất
check_df = pd.read_csv(OUTPUT_PATH)
assert list(check_df.columns) == list(sample_sub.columns), "Tên cột không khớp sample_submission.csv!"
assert len(check_df) == len(test_df), "Số dòng không khớp test.csv!"
assert (check_df["id"] == test_df["id"]).all(), "Thứ tự hoặc giá trị id không khớp test.csv!"
assert check_df["is_fraud"].isna().sum() == 0, "Có giá trị NaN trong dự đoán!"
assert (check_df["is_fraud"] >= 0.0).all() and (check_df["is_fraud"] <= 1.0).all(), "Xác suất nằm ngoài [0, 1]!"

print(f"Đã xuất thành công: {os.path.abspath(OUTPUT_PATH)}")
print(f"Kích thước tệp    : {check_df.shape}")
print("\n5 dòng đầu của submission.csv:")
print(check_df.head())
print("\nThống kê phân phối xác suất dự đoán:")
print(check_df["is_fraud"].describe())

Đã xuất thành công: F:\Năm 3\ML\PCCF\submission.csv
Kích thước tệp    : (299984, 2)

5 dòng đầu của submission.csv:
       id  is_fraud
0  699960  0.000391
1  699961  0.000391
2  699962  0.056760
3  699963  0.001114
4  699964  0.006872

Thống kê phân phối xác suất dự đoán:
count    299984.000000
mean          0.016759
std           0.067445
min           0.000391
25%           0.000391
50%           0.001416
75%           0.006872
max           0.953678
Name: is_fraud, dtype: float64
